In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
import sys
from logging import INFO, StreamHandler, getLogger

logger = getLogger()
if not logger.hasHandlers():
    logger.addHandler(StreamHandler(sys.stdout))
logger.setLevel(INFO)

# Import


In [ ]:
import itertools
import os
import pathlib
import sys
from collections import OrderedDict

import cartopy.crs as ccrs
import cftime
import matplotlib.pyplot as plt
import s3fs
import xarray as xr
from tqdm.notebook import tqdm

ROOT_DIR = str(pathlib.Path(".").resolve().parent.parent)
print(f"ROOT_DIR: {ROOT_DIR}")

In [ ]:
sys.path.append(ROOT_DIR)

In [ ]:
from src.util.datetime_util import get_target_datetimes, save_time_averaged_data

# Define constants


In [ ]:
CESM_DATA_DIR = f"{ROOT_DIR}/data/cesm_le"
os.makedirs(CESM_DATA_DIR, exist_ok=True)
CESM_DATA_DIR

# Download and save data


In [ ]:
s3 = s3fs.S3FileSystem(anon=True)

## Reference temerature


## 20C

In [ ]:
store_20C = s3fs.S3Map(
    root="s3://ncar-cesm-lens/atm/monthly/cesmLE-20C-TREFHT.zarr", s3=s3
)
ds_20C = xr.open_zarr(store_20C)["TREFHT"]

p = f"{CESM_DATA_DIR}/atm-monthly-cesmLE-20C-TREFHT.nc"
if not os.path.exists(p):
    ds_20C.to_netcdf(p)

## RCP85

In [ ]:
store_RCP85 = s3fs.S3Map(
    root="s3://ncar-cesm-lens/atm/monthly/cesmLE-RCP85-TREFHT.zarr", s3=s3
)
ds_RCP85 = xr.open_zarr(store_RCP85)["TREFHT"]

p = f"{CESM_DATA_DIR}/atm-monthly-cesmLE-RCP85-TREFHT.nc"
if not os.path.exists(p):
    ds_RCP85.to_netcdf(p)

## Precipitation


### 20C


In [ ]:
s3 = s3fs.S3FileSystem(anon=True)

store = s3fs.S3Map(root="s3://ncar-cesm-lens/atm/daily/cesmLE-20C-PRECT.zarr", s3=s3)
ds = xr.open_zarr(store)["PRECT"]
lst_dt = list(ds.time)

dir_path = f"{CESM_DATA_DIR}/atm-monthly-cesmLE-20C-PRECT"
os.makedirs(dir_path, exist_ok=True)

lst_ym = list(itertools.product(range(1920, 2006), range(1, 13)))
for year, month in tqdm(lst_ym):
    try:
        file_path = f"{dir_path}/cesmLE-20C-PRECT_{year:02}_{month:02}.nc"
        if os.path.exists(file_path):
            # logger.info(f"Skip because file already exists: {year=}, {month=}")
            continue
        target_dt = get_target_datetimes(lst_dt, year, month)
        save_time_averaged_data(ds, target_dt, file_path)
        logger.info(f"Finish saving: {year=}, {month=}")
    except Exception as e:
        print(file_path)
        print(e)

In [ ]:
saved_path = f"{CESM_DATA_DIR}/atm-monthly-cesmLE-20C-PRECT.nc"

if not os.path.exists(saved_path):
    dic = OrderedDict()
    for year, month in tqdm(lst_ym):
        try:
            file_path = f"{dir_path}/cesmLE-20C-PRECT_{year:02}_{month:02}.nc"
            ds = xr.open_dataset(file_path)
            dt = cftime.DatetimeNoLeap(
                year=year, month=month, day=14 if month == 2 else 15
            )
            dic[dt] = ds
        except Exception as e:
            print(file_path)
            print(e)

    combined_ds = xr.concat(
        list(dic.values()), xr.DataArray(list(dic.keys()), name="time", dims="time")
    )
    combined_ds = combined_ds.transpose("member_id", "time", "lat", "lon")
    combined_ds.to_netcdf(saved_path)
    logger.info("Finish saving the combined Dataset.")

### RCP85


In [ ]:
s3 = s3fs.S3FileSystem(anon=True)

store = s3fs.S3Map(root="s3://ncar-cesm-lens/atm/daily/cesmLE-RCP85-PRECT.zarr", s3=s3)
ds = xr.open_zarr(store)["PRECT"]
lst_dt = list(ds.time)

dir_path = f"{CESM_DATA_DIR}/atm-monthly-cesmLE-RCP85-PRECT"
os.makedirs(dir_path, exist_ok=True)

lst_ym = list(itertools.product(range(2006, 2101), range(1, 13)))
for year, month in tqdm(lst_ym):
    try:
        file_path = f"{dir_path}/cesmLE-RCP85-PRECT_{year:02}_{month:02}.nc"
        if os.path.exists(file_path):
            # logger.info(f"Skip because file already exists: {year=}, {month=}")
            continue
        target_dt = get_target_datetimes(lst_dt, year, month)
        save_time_averaged_data(ds, target_dt, file_path)
        logger.info(f"Finish saving: {year=}, {month=}")
    except Exception as e:
        print(file_path)
        print(e)

In [ ]:
saved_path = f"{CESM_DATA_DIR}/atm-monthly-cesmLE-RCP85-PRECT.nc"

if not os.path.exists(saved_path):
    dic = OrderedDict()
    for year, month in tqdm(lst_ym):
        try:
            file_path = f"{dir_path}/cesmLE-RCP85-PRECT_{year:02}_{month:02}.nc"
            ds = xr.open_dataset(file_path)
            dt = cftime.DatetimeNoLeap(
                year=year, month=month, day=14 if month == 2 else 15
            )
            dic[dt] = ds
        except Exception as e:
            print(file_path)
            print(e)

    combined_ds = xr.concat(
        list(dic.values()), xr.DataArray(list(dic.keys()), name="time", dims="time")
    )
    combined_ds = combined_ds.transpose("member_id", "time", "lat", "lon")
    combined_ds.to_netcdf(saved_path)
    logger.info("Finish saving the combined Dataset.")